In [17]:
!pip install pyspark --quiet
print('PySpark installation complete!')

PySpark installation complete!


In [18]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.functions import year,month,to_date,col,round as spark_round

import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

spark = SparkSession.builder \
        .appName('Day4_BigData_Sales') \
        .config('spark.sql.adaptive.enabled', 'true') \
        .getOrCreate()
print(f'spark version : {spark.version}')
print(f'Spark Session: ACTIVE')
print(f'Application : {spark.conf.get("spark.app.name")}')


spark version : 4.0.2
Spark Session: ACTIVE
Application : Day4_BigData_Sales


In [19]:
csv_file_path = '/content/drive/MyDrive/Internship/large_sales_data.csv'
df_sales = spark.read.csv(csv_file_path, header=True, inferSchema=True)


In [20]:
df_bronze= spark.read \
          .option('header', 'true') \
          .option('inferSchema', 'true') \
          .csv('/content/drive/MyDrive/Internship/large_sales_data.csv')
print(f'=== Bronze Layer -- Raw Data===')
print(f'Rows : {df_bronze.count()}')
print(f'Columns : {len(df_bronze.columns)}')
print(f'Names : {df_bronze.columns}')
print()

df_bronze.printSchema()

=== Bronze Layer -- Raw Data===
Rows : 5000
Columns : 13
Names : ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'revenue', 'order_date', 'city', 'region', 'sales_rep', 'payment_method', 'order_status']

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [21]:
print(f'Frist 5 rows :')
df_bronze.show(5, truncate=False)

print(f'\nBasic statistics for numeric columns: ')
df_bronze.select('quantity','unit_price','revenue').describe().show()


Frist 5 rows :
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat    |West  |Ravi K

In [22]:
from os.path import isfile
df_bronze.write \
        .mode('overwrite') \
        .parquet('sales_bronze.parquet')

print('Bronze Parquet saved: sales_bronze.praquet')

import os
def get_dir_size(dir_path):
    if os.path.isfile(dir_path):
        return os.path.getsize(dir_path) / 1024
    total = 0
    for dirpath,dirnames,filenames in os.walk(dir_path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total / 1024


csv_size = get_dir_size(csv_file_path)
parquet_size = get_dir_size('sales_bronze.parquet')
reduction = (1-parquet_size/csv_size)*100
print(f'CSV File Size: {csv_size:.2f} KB')
print(f'Parquet File Size: {parquet_size:.2f} KB')
print(f'Reduction in Size: {reduction:.1f}% smaller')
print(f'\nAt 1 TB scale: CSV=1000 GB -> parquet = {1000*(1-reduction/100):.0f}GB')

Bronze Parquet saved: sales_bronze.praquet
CSV File Size: 529.31 KB
Parquet File Size: 55.10 KB
Reduction in Size: 89.6% smaller

At 1 TB scale: CSV=1000 GB -> parquet = 104GB


In [23]:
df_silver = df_bronze \
           .dropDuplicates() \
           .dropna(subset=['order_id','product','revenue'])

df_silver = df_silver.withColumn(
    'order_date',
    to_date(col('order_date'), 'yyyy-MM-dd')
)

df_silver= df_silver \
          .withColumn('order_year', year(col('order_date'))) \
          .withColumn('order_month', month(col('order_date')))

df_silver = df_silver.withColumn(
    'revenue_category',
    f.when(col('revenue') > 40000, 'High')
    .when(col('revenue') > 10000, 'Medium')
    .otherwise('Low')
)


print(f'Silver Layer Rows : {df_silver.count()}')
print('New columns added: order_year, order_month, revenue_category')
df_silver.select('product','revenue','order_year','order_month','revenue_category').show(5)

Silver Layer Rows : 5000
New columns added: order_year, order_month, revenue_category
+--------+-------+----------+-----------+----------------+
| product|revenue|order_year|order_month|revenue_category|
+--------+-------+----------+-----------+----------------+
|Keyboard|  13200|      2023|          2|          Medium|
|  Webcam|  17500|      2023|          1|          Medium|
| Speaker|  58500|      2023|          4|            High|
|Keyboard|   9600|      2023|         12|             Low|
|  Laptop| 180000|      2023|          8|            High|
+--------+-------+----------+-----------+----------------+
only showing top 5 rows


In [24]:
df_silver.write \
         .mode('overwrite') \
         .parquet('sales_silver.parquet')

print('Silver Parquet saved: sales_silver.praquet')
print(f'Silver Size: {get_dir_size("sales_silver.parquet"):.1f} KB')

df_verify = spark.read.parquet('sales_silver.parquet')
print(f'Verify Silver Size')
print(f'Read-back rows: {df_verify.count()} (should match silver count)')
df_verify.printSchema()

Silver Parquet saved: sales_silver.praquet
Silver Size: 59.8 KB
Verify Silver Size
Read-back rows: 5000 (should match silver count)
root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- revenue_category: string (nullable = true)

